# EEG Multiclass Classification

This notebook is a compact project walkthrough. The reusable implementation lives in `src/`; use this notebook to inspect the data, run the official pipeline, and review generated artifacts.

## 1. Imports and Paths

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.data import clean_dataframe, feature_columns, load_raw_data, split_and_scale
from src.evaluation import CLASS_NAMES

ROOT = Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "data.csv"
OUTPUTS_DIR = ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
MODELS_DIR = OUTPUTS_DIR / "models"

SEED = 42
CLASS_NAMES

## 2. Data Loading and Preprocessing Preview

The full preprocessing implementation is in `src/data.py`. The preview below verifies the raw shape, class mapping, split sizes, and leakage-safe train-only scaling behavior.

In [ ]:
raw = load_raw_data(RAW_PATH)
cleaned = clean_dataframe(raw)
splits = split_and_scale(cleaned, random_state=SEED)

pd.DataFrame(
    {
        "dataset": ["raw", "cleaned", "train", "validation", "test"],
        "rows": [len(raw), len(cleaned), len(splits.y_train), len(splits.y_val), len(splits.y_test)],
        "features": [len(feature_columns(cleaned)), len(feature_columns(cleaned)), splits.x_train.shape[1], splits.x_val.shape[1], splits.x_test.shape[1]],
    }
)

In [ ]:
pd.DataFrame(
    {
        "raw_y": sorted(cleaned["y"].unique()),
        "model_target": sorted(cleaned["target"].unique()),
        "class_name": CLASS_NAMES,
    }
)

## 3. Official Reproducible Run

Run the command below to regenerate processed data, Optuna results, the final model, calibration, uncertainty tables, PFI, LIME, figures, and metrics.

In [ ]:
# This is intentionally left commented because it runs the full 30-trial experiment.
# %run -m src.train --seed 42 --n-trials 30 --max-epochs 35 --force-hpo

## 4. Hyperparameter Search

In [ ]:
hpo = pd.read_csv(TABLES_DIR / "hyperparameter_search.csv")
best_params = pd.read_json(MODELS_DIR / "best_params.json", typ="series")

print(f"Trials: {len(hpo)}")
display(hpo.sort_values("value", ascending=False).head(10))
display(best_params)

## 5. Test Metrics

In [ ]:
metrics = pd.read_csv(METRICS_DIR / "test_metrics.csv")
classification_report = pd.read_csv(METRICS_DIR / "classification_report.csv", index_col=0)

display(metrics)
display(classification_report)

## 6. Evaluation Figures

In [ ]:
for figure_name in ["confusion_matrix.png", "reliability_diagram.png", "pfi_test.png", "lime_global.png"]:
    display(Image(filename=str(FIGURES_DIR / figure_name)))

## 7. Uncertainty and Explainability Tables

In [ ]:
uncertainty = pd.read_csv(TABLES_DIR / "uncertainty_all_test_predictions.csv")
pfi_test = pd.read_csv(TABLES_DIR / "pfi_test.csv")
lime_global = pd.read_csv(TABLES_DIR / "lime_global.csv")

display(uncertainty.sort_values("entropy", ascending=False).head(10))
display(pfi_test.head(10))
display(lime_global.head(10))

## 8. Notes

Model weights, Lightning checkpoints, scalers, and calibration joblibs are generated locally and ignored by Git. Small JSON run metadata and all CSV/figure explanation artifacts are kept for review.